In [1]:
import pandas as pd

df = pd.read_excel("15000politician_statements.xlsx")

property_dict = {}
for _, row in df.iterrows():
    proid   = row.get('Property_ID',    '')
    prolabel = row.get('Property_Label', '')
    if isinstance(proid, str) and proid.strip() != '':
        property_dict[proid.strip()] = prolabel

result_df = pd.DataFrame(list(property_dict.items()),
                         columns=['Property_ID', 'Property_Label'])
result_df = result_df.sort_values('Property_ID').reset_index(drop=True)

print(f"Number of unique property types: {len(result_df)}")
result_df.to_excel("15000_politician_property_list.xlsx", index=False)
print("Saved to 15000_politician_property_list.xlsx")

Number of unique property types: 1759
Saved to 15000_politician_property_list.xlsx


In [2]:
import requests
import time

headers = {
    'User-Agent': 'WikidataResearch/1.0 (academic research; your_email@example.com)'
}

def get_descriptions_batch(proid_list):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": "|".join(proid_list),
        "props": "descriptions",
        "languages": "en",
        "format": "json"
    }
    resp = requests.get(url, params=params, headers=headers, timeout=30)
    return resp.json()

proids = result_df['Property_ID'].tolist()
batches = [proids[i:i+50] for i in range(0, len(proids), 50)]
desc_map = {}

for batch in batches:
    data = get_descriptions_batch(batch)
    if data and 'entities' in data:
        for proid, entity in data['entities'].items():
            desc = entity.get('descriptions', {}).get('en', {}).get('value', '')
            desc_map[proid] = desc
    time.sleep(0.3)

result_df['Description'] = result_df['Property_ID'].map(desc_map).fillna('')
result_df.to_excel("15000politician_property_list.xlsx", index=False)
print("Saved to 15000_politician_property_list.xlsx")

Saved to 15000_politician_property_list.xlsx
